In [2]:
#!/usr/bin/env python3
# ============================================================
# Coma cluster "one-shot" run (scalar self-energy included)
#
# PURPOSE:
#   Integrate the scalar-augmented hydrostatic ODE and print:
#     - M_g, M_phi, M_total at r ~ R500 (set by X_MAX)
#     - M_phi/M_g
#     - outer gas slope (should be ~ -2)
#
# CHOSEN CLUSTER: Coma
#   T0_keV ≈ 8.2 keV
#   R500 ≈ 1.30 Mpc
#   Gas mass target at R500 ≈ 1.4e14 Msun
#
# ANALYTIC SHOOT (rough):
#   Your previous run with Y0_CENTRAL=1 gave M_g ~ 3.0e14 Msun at ~R500.
#   So use Y0_CENTRAL ~ 1.4/3.0 ≈ 0.47.
#
# NOTE:
#   This script does NOT auto-shoot; it uses the analytic estimate Y0_CENTRAL.
# ============================================================

import math
import numpy as np
from scipy.integrate import solve_ivp

# -----------------------------
# Physical constants (SI)
# -----------------------------
G   = 6.674e-11
a0  = 1.2e-10
kB  = 1.380649e-23
m_p = 1.67262192369e-27
mu_gas = 0.6

KPC_M   = 3.085677581e19
MSUN_KG = 1.98847e30

# -----------------------------
# Coma inputs / run knobs
# -----------------------------
T0_keV      = 8.2          # Coma-like temperature
R500_kpc    = 1300.0       # Coma R500 ~ 1.30 Mpc
Y0_CENTRAL  = 0.32       # analytic shoot estimate
X0          = 1e-5         # start x = r/r0
RTOL        = 1e-8
ATOL        = 1e-12
MAX_STEP    = 0.2
U_SERIES    = 2e-3         # series threshold for F(Y) stability
PLOT        = False        # optional plots

# -----------------------------
# Temperature profile (dimensionless): theta(x) = T/T0
# Default: isothermal.
# -----------------------------
def theta(x: float) -> float:
    return 1.0

def dtheta_dx(x: float) -> float:
    return 0.0

# -----------------------------
# Scalar functions
#   mu(Y) = 1 - exp(-Y^(1/4))
#   F(Y)  = ∫_0^Y mu(s) ds (stable: series for small U=Y^(1/4), exact otherwise)
#   y_phi(Y) = 0.5*(2Y*mu(Y) - F(Y))
# -----------------------------
def U_from_Y(Y: float) -> float:
    if Y <= 0.0:
        return 0.0
    return math.exp(0.25 * math.log(Y))

def mu_Y(Y: float) -> float:
    U = U_from_Y(Y)
    return -math.expm1(-U)  # stable for small U

def F_Y(Y: float) -> float:
    if Y <= 0.0:
        return 0.0
    U = U_from_Y(Y)

    # series for small U to avoid catastrophic cancellation
    if U < U_SERIES:
        U2 = U * U
        U4 = U2 * U2
        U5 = U4 * U
        U6 = U5 * U
        U7 = U6 * U
        U8 = U7 * U
        U9 = U8 * U
        U10 = U9 * U
        # F(Y) = ∫_0^U 4 t^3(1-e^{-t}) dt
        return 4.0 * (
            (U5 / 5.0)
            - (U6 / 12.0)
            + (U7 / 42.0)
            - (U8 / 192.0)
            + (U9 / 1080.0)
            - (U10 / 7200.0)
        )

    # exact analytic for moderate/large U:
    # F(Y) = Y - 24 + 4 e^{-U}(U^3+3U^2+6U+6)
    e = math.exp(-U)
    poly = (U**3 + 3.0 * U**2 + 6.0 * U + 6.0)
    return Y - 24.0 + 4.0 * e * poly

def y_phi_from_Y(Y: float) -> float:
    if Y <= 0.0:
        return 0.0
    mu = mu_Y(Y)
    FY = F_Y(Y)
    val = 0.5 * (2.0 * Y * mu - FY)
    if val < 0.0 and val > -1e-14:
        return 0.0
    return max(0.0, val)

# -----------------------------
# Scaling (correct)
#   r0 = (kB*T0)/(mu*m_p*a0) [m]
#   M0 = a0*r0^2/G           [kg]
#   rho0 = a0/(4π G r0)      [kg/m^3]
# -----------------------------
T0_K = T0_keV * 1.16045e7
r0   = (kB * T0_K) / (mu_gas * m_p * a0)
M0   = (a0 * r0 * r0) / G
rho0 = a0 / (4.0 * math.pi * G * r0)

# Choose X_MAX so that r_final ~ R500
R500_m = R500_kpc * KPC_M
X_MAX  = R500_m / r0

print("=== COMA SCALAR-AUGMENTED HYDROSTATIC RUN ===")
print(f"T0_keV      = {T0_keV:.3f}  (T0_K = {T0_K:.6e} K)")
print(f"r0          = {r0/KPC_M:.3f} kpc")
print(f"R500        = {R500_kpc:.1f} kpc  -> X_MAX = {X_MAX:.6f}")
print(f"rho0        = {rho0:.6e} kg/m^3")
print(f"M0          = {M0/MSUN_KG:.6e} Msun")
print(f"Y0_CENTRAL  = {Y0_CENTRAL:.6f}")
print()

# -----------------------------
# ODE in terms of [ln y, mg, mp]
# to avoid stiffness/underflow in y.
# -----------------------------
def rhs(x: float, u: np.ndarray) -> np.ndarray:
    ln_y, mg, mp = float(u[0]), float(u[1]), float(u[2])

    if x <= 0.0 or not (math.isfinite(ln_y) and math.isfinite(mg) and math.isfinite(mp)):
        return np.array([0.0, 0.0, 0.0], dtype=np.float64)

    # Enforce nonnegativity
    mg = max(mg, 0.0)
    mp = max(mp, 0.0)

    y = math.exp(ln_y)
    m_tot = mg + mp

    # s = sqrt(m_tot)/x ; denom = 1-exp(-s)
    s = math.sqrt(max(m_tot, 0.0)) / x
    denom = -math.expm1(-s)
    if denom < 1e-14:
        denom = max(s, 1e-14)

    # ghat = g/a0 (dimensionless)
    ghat = (m_tot / (x * x)) / denom

    # scalar density closure
    Y = ghat * ghat
    yphi = y_phi_from_Y(Y)

    th = theta(x)
    dth = dtheta_dx(x)

    # d ln y / dx
    dlny_dx = -(ghat / th + dth / th)

    # mass ODEs
    dmg_dx = x * x * y
    dmp_dx = x * x * yphi

    return np.array([dlny_dx, dmg_dx, dmp_dx], dtype=np.float64)

# -----------------------------
# Initial conditions
# -----------------------------
y0 = float(Y0_CENTRAL)
lny0 = math.log(max(y0, 1e-300))
mg0 = (X0**3) * y0 / 3.0
mp0 = 0.0
u0 = np.array([lny0, mg0, mp0], dtype=np.float64)

# -----------------------------
# Integrate
# -----------------------------
sol = solve_ivp(
    rhs,
    (X0, X_MAX),
    u0,
    method="Radau",
    rtol=RTOL,
    atol=ATOL,
    max_step=MAX_STEP
)

if not sol.success:
    raise RuntimeError("ODE integration failed: " + str(sol.message))

x    = sol.t
ln_y = sol.y[0]
y    = np.exp(ln_y)
mg   = sol.y[1]
mp   = sol.y[2]
m_tot = mg + mp

# -----------------------------
# Report at outer radius (≈ R500)
# -----------------------------
r_final_kpc = (x[-1] * r0) / KPC_M
Mg_Msun     = mg[-1] * (M0 / MSUN_KG)
Mphi_Msun   = mp[-1] * (M0 / MSUN_KG)
Mtot_Msun   = m_tot[-1] * (M0 / MSUN_KG)

print("=== RESULTS at r ≈ R500 ===")
print(f"x_final           = {x[-1]:.6f}")
print(f"r_final           = {r_final_kpc:.3f} kpc")
print(f"M_g               = {Mg_Msun:.3e} Msun")
print(f"M_phi             = {Mphi_Msun:.3e} Msun")
print(f"M_total           = {Mtot_Msun:.3e} Msun")
print(f"M_phi / M_g       = {(Mphi_Msun/Mg_Msun):.6f}")
print()

# -----------------------------
# Outer slope check: d ln rho / d ln r near R500
# -----------------------------
# Use last ~10% of points (robust median)
n = len(x)
j0 = max(0, int(0.9 * n))
slope = np.gradient(np.log(y + 1e-300), np.log(x + 1e-300))
outer_slope = float(np.median(slope[j0:]))

print("=== CONSISTENCY CHECKS ===")
print(f"outer slope median d ln rho / d ln r  ≈ {outer_slope:.3f}  (target ~ -2)")
print(f"rho_g(R500)/rho0                      = {y[-1]:.6e}")
print()

if PLOT:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6,4))
    plt.loglog(x, mg, label="M_g/M0")
    plt.loglog(x, mp, label="M_phi/M0")
    plt.loglog(x, m_tot, "--", label="M_tot/M0")
    plt.xlabel("x = r/r0")
    plt.ylabel("Enclosed mass (dimensionless)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6,4))
    plt.loglog(x, y, label="rho_g/rho0")
    plt.xlabel("x = r/r0")
    plt.ylabel("Gas density (dimensionless)")
    plt.legend()
    plt.tight_layout()
    plt.show()

=== COMA SCALAR-AUGMENTED HYDROSTATIC RUN ===
T0_keV      = 8.200  (T0_K = 9.515690e+07 K)
r0          = 353.543 kpc
R500        = 1300.0 kpc  -> X_MAX = 3.677059
rho0        = 1.311571e-23 kg/m^3
M0          = 1.076125e+14 Msun
Y0_CENTRAL  = 0.320000

=== RESULTS at r ≈ R500 ===
x_final           = 3.677059
r_final           = 1300.000 kpc
M_g               = 2.007e+14 Msun
M_phi             = 1.871e+14 Msun
M_total           = 3.878e+14 Msun
M_phi / M_g       = 0.932581

=== CONSISTENCY CHECKS ===
outer slope median d ln rho / d ln r  ≈ -1.971  (target ~ -2)
rho_g(R500)/rho0                      = 5.986417e-02



In [3]:
#!/usr/bin/env python3
# ============================================================
# Coma cluster run with DECLINING T(r) (Vikhlinin-type)
# ============================================================

import math
import numpy as np
from scipy.integrate import solve_ivp

# -----------------------------
# Physical constants (SI)
# -----------------------------
G   = 6.674e-11
a0  = 1.2e-10
kB  = 1.380649e-23
m_p = 1.67262192369e-27
mu_gas = 0.6

KPC_M   = 3.085677581e19
MSUN_KG = 1.98847e30

# -----------------------------
# Coma inputs
# -----------------------------
T0_keV     = 8.2
R500_kpc   = 1300.0
Y0_CENTRAL = 0.32
X0         = 1e-5
RTOL       = 1e-8
ATOL       = 1e-12
MAX_STEP   = 0.2
U_SERIES   = 2e-3

# -----------------------------
# Scaling
# -----------------------------
T0_K = T0_keV * 1.16045e7
r0   = (kB * T0_K) / (mu_gas * m_p * a0)
M0   = (a0 * r0 * r0) / G
rho0 = a0 / (4.0 * math.pi * G * r0)

R500_m = R500_kpc * KPC_M
X_MAX  = R500_m / r0

# -----------------------------
# Temperature profile (Vikhlinin-type)
# -----------------------------
x_t = 0.45 * X_MAX   # turnover radius ~0.45 R500
a_T = 0.0
b_T = 2.0
c_T = 1.0

def theta(x):
    if x <= 0:
        return 1.0
    return (x/x_t)**(-a_T) / (1.0 + (x/x_t)**b_T)**(c_T/b_T)

def dtheta_dx(x):
    if x <= 0:
        return 0.0
    num = -(c_T)*(x/x_t)**(b_T-1)
    den = x_t*(1.0 + (x/x_t)**b_T)
    return theta(x) * num/den

# -----------------------------
# Scalar functions
# -----------------------------
def U_from_Y(Y):
    if Y <= 0.0:
        return 0.0
    return math.exp(0.25 * math.log(Y))

def mu_Y(Y):
    U = U_from_Y(Y)
    return -math.expm1(-U)

def F_Y(Y):
    if Y <= 0.0:
        return 0.0
    U = U_from_Y(Y)
    if U < U_SERIES:
        U2 = U*U; U4 = U2*U2
        U5 = U4*U; U6 = U5*U
        U7 = U6*U; U8 = U7*U
        U9 = U8*U; U10 = U9*U
        return 4.0*(U5/5 - U6/12 + U7/42 - U8/192 + U9/1080 - U10/7200)
    e = math.exp(-U)
    poly = (U**3 + 3*U**2 + 6*U + 6)
    return Y - 24.0 + 4.0*e*poly

def y_phi_from_Y(Y):
    if Y <= 0.0:
        return 0.0
    mu = mu_Y(Y)
    FY = F_Y(Y)
    return max(0.0, 0.5*(2*Y*mu - FY))

# -----------------------------
# ODE system: u=[ln y, mg, mp]
# -----------------------------
def rhs(x, u):
    ln_y, mg, mp = u
    if x <= 0:
        return [0,0,0]

    y = math.exp(ln_y)
    m_tot = max(mg + mp, 0.0)

    s = math.sqrt(m_tot)/x
    denom = -math.expm1(-s)
    if denom < 1e-14:
        denom = max(s, 1e-14)

    ghat = (m_tot/x**2)/denom
    Y = ghat*ghat
    yphi = y_phi_from_Y(Y)

    th = theta(x)
    dth = dtheta_dx(x)

    dlny_dx = -(ghat/th + dth/th)
    dmg_dx  = x*x*y
    dmp_dx  = x*x*yphi

    return [dlny_dx, dmg_dx, dmp_dx]

# -----------------------------
# Initial conditions
# -----------------------------
y0 = Y0_CENTRAL
u0 = [math.log(y0), (X0**3)*y0/3.0, 0.0]

# -----------------------------
# Integrate
# -----------------------------
sol = solve_ivp(rhs, (X0, X_MAX), u0, method="Radau",
                rtol=RTOL, atol=ATOL, max_step=MAX_STEP)

if not sol.success:
    raise RuntimeError(sol.message)

x  = sol.t
y  = np.exp(sol.y[0])
mg = sol.y[1]
mp = sol.y[2]

Mtot = (mg[-1]+mp[-1])*(M0/MSUN_KG)
Mg   = mg[-1]*(M0/MSUN_KG)
Mphi = mp[-1]*(M0/MSUN_KG)

print("\n=== COMA with declining T(r) ===")
print(f"r_final = {x[-1]*r0/KPC_M:.1f} kpc")
print(f"M_g     = {Mg:.3e} Msun")
print(f"M_phi   = {Mphi:.3e} Msun")
print(f"M_total = {Mtot:.3e} Msun")
print(f"M_phi/M_g = {Mphi/Mg:.3f}")


=== COMA with declining T(r) ===
r_final = 1300.0 kpc
M_g     = 2.036e+14 Msun
M_phi   = 2.369e+14 Msun
M_total = 4.405e+14 Msun
M_phi/M_g = 1.163


In [4]:
#!/usr/bin/env python3
# ============================================================
# Coma cluster run with DECLINING T(r) (robust)
# ============================================================

import math
import numpy as np
from scipy.integrate import solve_ivp

# -----------------------------
# Physical constants (SI)
# -----------------------------
G   = 6.674e-11
a0  = 1.2e-10
kB  = 1.380649e-23
m_p = 1.67262192369e-27
mu_gas = 0.6

KPC_M   = 3.085677581e19
MSUN_KG = 1.98847e30

# -----------------------------
# Coma inputs
# -----------------------------
T0_keV     = 8.2
R500_kpc   = 1300.0
Y0_CENTRAL = 0.32

X0       = 1e-5
RTOL     = 1e-8
ATOL     = 1e-12
MAX_STEP = 0.2
U_SERIES = 2e-3

# Numerical guards
LN_Y_FLOOR = -120.0   # exp(-120) ~ 7e-53, effectively zero but finite
Y_FLOOR    = 1e-80    # for safe logs/ratios

# -----------------------------
# Scaling
# -----------------------------
T0_K = T0_keV * 1.16045e7
r0   = (kB * T0_K) / (mu_gas * m_p * a0)              # [m]
M0   = (a0 * r0 * r0) / G                             # [kg]
rho0 = a0 / (4.0 * math.pi * G * r0)                  # [kg/m^3]

R500_m = R500_kpc * KPC_M
X_MAX  = R500_m / r0

# -----------------------------
# Temperature profile (Vikhlinin-like, conservative)
#   theta(x)=T/T0
#
# Choose parameters so that theta(R500) ~ 0.7–0.8 (Coma-like).
# -----------------------------
x_t = 0.45 * X_MAX   # turnover radius ~0.45 R500
a_T = 0.0
b_T = 2.0
c_T = 0.8            # slightly softer than 1.0 to avoid over-steepening

def theta(x: float) -> float:
    if x <= 0.0:
        return 1.0
    z = (x / x_t)
    # theta = z^{-a} * (1+z^b)^(-c/b)
    return (z ** (-a_T)) * ((1.0 + z**b_T) ** (-c_T / b_T))

def dtheta_dx(x: float) -> float:
    if x <= 0.0:
        return 0.0
    z = x / x_t
    th = theta(x)
    # d/dx ln theta = -(a/x)  - (c/b) * d/dx ln(1+z^b)
    # here a=0 so first term drops; keep full formula anyway
    term_a = -a_T / x if a_T != 0.0 else 0.0
    dln = term_a - (c_T / b_T) * ( (b_T * z**(b_T-1)) / (x_t * (1.0 + z

SyntaxError: incomplete input (ipython-input-1046841064.py, line 76)

In [5]:
#!/usr/bin/env python3
# ============================================================
# COMA — Scalar-augmented hydrostatic ODE (NEWEST, HARDENED)
#
# What this script does:
#   - Solves hydrostatic equilibrium for hot ICM gas in a modified-gravity model
#     with scalar self-energy included as an effective mass component.
#   - Uses a declining temperature profile theta(x)=T/T0 (Vikhlinin-like).
#   - Integrates stiff ODEs robustly (Radau) evolving ln(y) to avoid underflow.
#
# Outputs at r = R500:
#   - M_g, M_phi, M_total
#   - M_phi/M_g
#   - outer density slope d ln rho / d ln r (should be ~ -2)
#
# Notes:
#   - Y0_CENTRAL is a SHOOTING parameter (central density scale).
#   - Default is the analytically suggested range for Coma. Adjust if needed.
# ============================================================

import math
import numpy as np
from scipy.integrate import solve_ivp

# -----------------------------
# Physical constants (SI)
# -----------------------------
G   = 6.674e-11
a0  = 1.2e-10
kB  = 1.380649e-23
m_p = 1.67262192369e-27
mu_gas = 0.6

KPC_M   = 3.085677581e19
MSUN_KG = 1.98847e30

# -----------------------------
# Coma inputs / run knobs
# -----------------------------
T0_keV     = 8.2
R500_kpc   = 1300.0

# SHOOTING PARAMETER (try 0.23, 0.28, 0.32)
Y0_CENTRAL = 0.23

# ODE integration knobs
X0       = 1e-5
RTOL     = 1e-8
ATOL     = 1e-12
MAX_STEP = 0.2

# Scalar primitive stability
U_SERIES = 2e-3

# Numerical guards
LN_Y_FLOOR = -200.0
DENOM_FLOOR = 1e-14

PLOT = False

# -----------------------------
# Scaling (correct)
#   r0 = (kB*T0)/(mu*m_p*a0) [m]
#   M0 = a0*r0^2/G           [kg]
#   rho0 = a0/(4π G r0)      [kg/m^3]
# -----------------------------
T0_K = T0_keV * 1.16045e7
r0   = (kB * T0_K) / (mu_gas * m_p * a0)              # [m]
M0   = (a0 * r0 * r0) / G                             # [kg]
rho0 = a0 / (4.0 * math.pi * G * r0)                  # [kg/m^3]

R500_m = R500_kpc * KPC_M
X_MAX  = R500_m / r0

# -----------------------------
# Temperature profile theta(x)=T/T0 (declining, conservative)
# Choose parameters so theta(R500) ~ 0.7–0.8.
# -----------------------------
x_t = 0.45 * X_MAX
a_T = 0.0
b_T = 2.0
c_T = 0.8

def theta(x: float) -> float:
    if x <= 0.0:
        return 1.0
    z = x / x_t
    return (z ** (-a_T)) * ((1.0 + z**b_T) ** (-c_T / b_T))

def dtheta_dx(x: float) -> float:
    if x <= 0.0:
        return 0.0
    z = x / x_t
    th = theta(x)
    # d/dx ln theta = -(a/x) - (c/b)* d/dx ln(1+z^b)
    term_a = -a_T / x if a_T != 0.0 else 0.0
    dln = term_a - (c_T / b_T) * ((b_T * z**(b_T - 1.0)) / (x_t * (1.0 + z**b_T)))
    return th * dln

# -----------------------------
# Scalar functions
#   mu(Y)=1-exp(-Y^(1/4))
#   F(Y)=∫_0^Y mu(s) ds  (stable series for small U=Y^(1/4))
#   y_phi(Y)=0.5*(2Y*mu - F)
# -----------------------------
def U_from_Y(Y: float) -> float:
    if Y <= 0.0:
        return 0.0
    return math.exp(0.25 * math.log(Y))

def mu_Y(Y: float) -> float:
    U = U_from_Y(Y)
    return -math.expm1(-U)

def F_Y(Y: float) -> float:
    if Y <= 0.0:
        return 0.0
    U = U_from_Y(Y)

    if U < U_SERIES:
        U2 = U * U
        U4 = U2 * U2
        U5 = U4 * U
        U6 = U5 * U
        U7 = U6 * U
        U8 = U7 * U
        U9 = U8 * U
        U10 = U9 * U
        return 4.0 * (
            (U5 / 5.0)
            - (U6 / 12.0)
            + (U7 / 42.0)
            - (U8 / 192.0)
            + (U9 / 1080.0)
            - (U10 / 7200.0)
        )

    e = math.exp(-U)
    poly = (U**3 + 3.0 * U**2 + 6.0 * U + 6.0)
    return Y - 24.0 + 4.0 * e * poly

def y_phi_from_Y(Y: float) -> float:
    if Y <= 0.0:
        return 0.0
    mu = mu_Y(Y)
    FY = F_Y(Y)
    val = 0.5 * (2.0 * Y * mu - FY)
    if val < 0.0 and val > -1e-14:
        return 0.0
    return max(0.0, val)

# -----------------------------
# ODE system: u=[ln y, mg, mp]
#   y = rho_g/rho0
#   mg = M_g/M0
#   mp = M_phi/M0
# -----------------------------
def rhs(x: float, u: np.ndarray) -> np.ndarray:
    ln_y, mg, mp = float(u[0]), float(u[1]), float(u[2])

    if x <= 0.0 or not (math.isfinite(ln_y) and math.isfinite(mg) and math.isfinite(mp)):
        return np.array([0.0, 0.0, 0.0], dtype=np.float64)

    ln_y = max(ln_y, LN_Y_FLOOR)   # prevent exp underflow to 0
    y = math.exp(ln_y)

    mg = max(mg, 0.0)
    mp = max(mp, 0.0)
    m_tot = mg + mp

    # s = sqrt(m_tot)/x; denom = 1-exp(-s)
    s = math.sqrt(max(m_tot, 0.0)) / x
    denom = -math.expm1(-s)
    if denom < DENOM_FLOOR:
        denom = max(s, DENOM_FLOOR)

    # ghat = g/a0
    ghat = (m_tot / (x * x)) / denom
    Y = ghat * ghat
    yphi = y_phi_from_Y(Y)

    th = theta(x)
    dth = dtheta_dx(x)

    dlny_dx = -(ghat / th + dth / th)
    dmg_dx  = x * x * y
    dmp_dx  = x * x * yphi

    return np.array([dlny_dx, dmg_dx, dmp_dx], dtype=np.float64)

# -----------------------------
# Initial conditions
# -----------------------------
y0 = float(Y0_CENTRAL)
lny0 = math.log(max(y0, 1e-300))
mg0 = (X0**3) * y0 / 3.0
mp0 = 0.0
u0 = np.array([lny0, mg0, mp0], dtype=np.float64)

# -----------------------------
# Run
# -----------------------------
print("=== COMA — NEWEST HARDENED RUN ===")
print(f"T0_keV      = {T0_keV:.3f}")
print(f"R500_kpc    = {R500_kpc:.1f}")
print(f"r0          = {r0/KPC_M:.3f} kpc")
print(f"X_MAX       = {X_MAX:.6f}")
print(f"M0          = {M0/MSUN_KG:.3e} Msun")
print(f"rho0        = {rho0:.3e} kg/m^3")
print(f"Y0_CENTRAL  = {Y0_CENTRAL:.6f}")
print(f"theta(R500) = {theta(X_MAX):.4f}")
print()

sol = solve_ivp(
    rhs,
    (X0, X_MAX),
    u0,
    method="Radau",
    rtol=RTOL,
    atol=ATOL,
    max_step=MAX_STEP
)

if not sol.success:
    raise RuntimeError("ODE integration failed: " + str(sol.message))

x    = sol.t
ln_y = sol.y[0]
y    = np.exp(np.maximum(ln_y, LN_Y_FLOOR))
mg   = sol.y[1]
mp   = sol.y[2]
m_tot = mg + mp

# Physical outputs at R500
r_final_kpc = (x[-1] * r0) / KPC_M
Mg_Msun     = mg[-1] * (M0 / MSUN_KG)
Mphi_Msun   = mp[-1] * (M0 / MSUN_KG)
Mtot_Msun   = (mg[-1] + mp[-1]) * (M0 / MSUN_KG)

print("=== RESULTS at r ≈ R500 ===")
print(f"r_final      = {r_final_kpc:.1f} kpc")
print(f"M_g          = {Mg_Msun:.3e} Msun")
print(f"M_phi        = {Mphi_Msun:.3e} Msun")
print(f"M_total      = {Mtot_Msun:.3e} Msun")
print(f"M_phi/M_g    = {(Mphi_Msun/Mg_Msun):.6f}")
print()

# Outer slope check
slope = np.gradient(np.log(y + 1e-300), np.log(x + 1e-300))
outer_slope = float(np.median(slope[int(0.9*len(slope)):]))

print("=== CHECKS ===")
print(f"outer slope median d ln rho / d ln r ≈ {outer_slope:.3f} (target ~ -2)")
print(f"rho_g(R500)/rho0                     = {y[-1]:.6e}")
print()

if PLOT:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6,4))
    plt.loglog(x, mg, label="M_g/M0")
    plt.loglog(x, mp, label="M_phi/M0")
    plt.loglog(x, m_tot, "--", label="M_tot/M0")
    plt.xlabel("x = r/r0")
    plt.ylabel("Enclosed mass (dimensionless)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6,4))
    plt.loglog(x, y, label="rho_g/rho0")
    plt.xlabel("x = r/r0")
    plt.ylabel("rho_g/rho0")
    plt.legend()
    plt.tight_layout()
    plt.show()

=== COMA — NEWEST HARDENED RUN ===
T0_keV      = 8.200
R500_kpc    = 1300.0
r0          = 353.543 kpc
X_MAX       = 3.677059
M0          = 1.076e+14 Msun
rho0        = 1.312e-23 kg/m^3
Y0_CENTRAL  = 0.230000
theta(R500) = 0.4904

=== RESULTS at r ≈ R500 ===
r_final      = 1300.0 kpc
M_g          = 1.828e+14 Msun
M_phi        = 1.610e+14 Msun
M_total      = 3.438e+14 Msun
M_phi/M_g    = 0.880308

=== CHECKS ===
outer slope median d ln rho / d ln r ≈ -2.614 (target ~ -2)
rho_g(R500)/rho0                     = 4.438514e-02

